**ASSIGNMENT NLP – 5 (Token Classification: POS Tagging & Chunking)**

**Step 1 : Install Required Libraries**

In [ ]:
!pip install transformers datasets seqeval torch

In [ ]:
!pip install --upgrade transformers

In [ ]:
! pip install datasets==2.16.1

**Import libraries**

In [ ]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

**Load the datset**

In [ ]:
dataset = load_dataset("conll2003")

**Label Categories**

In [ ]:
pos_labels = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

pos_labels[:10]

['"', "''", '#', '$', '(', ')', ',', '.', ':', '``']

**Data Preprocessing(Tokenization + Alignment)**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

**POS Tokenization**

In [ ]:
def tokenize_pos(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

**Apply Preprocessing**

In [ ]:
pos_tokenized = dataset.map(tokenize_pos, batched=True)
small_train = pos_tokenized["train"].select(range(1000))
small_val = pos_tokenized["validation"].select(range(200))

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(pos_labels)
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

**Training**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="output",
    logging_steps=50,   #  increase this (default is small)
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator
)

predictions = trainer.predict(small_val)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=2)
labels = predictions.label_ids

In [ ]:
true_labels = []
true_predictions = []

for pred, label in zip(preds, labels):
    true_labels.append([pos_labels[l] for l in label if l != -100])
    true_predictions.append([pos_labels[p] for p, l in zip(pred, label) if l != -100])

In [ ]:
from seqeval.metrics import classification_report

In [ ]:
# Inference
sentence = "John works at Google in California"
words = sentence.split()

inputs = tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True
)

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

word_ids = inputs.word_ids()

predicted_labels = []
previous_word_idx = None

for idx, word_idx in enumerate(word_ids):
    if word_idx is None:
        continue
    elif word_idx != previous_word_idx:
        predicted_labels.append(pos_labels[predictions[0][idx].item()])
    previous_word_idx = word_idx

for word, label in zip(words, predicted_labels):
    print(word, "->", label)

John -> WP
works -> LS
at -> TO
Google -> CD
in -> TO
California -> NNS
